#### This notebook looks at temperature-dependent changes to embryo morphology

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
from glob2 import glob
from src.functions.plot_functions import format_3d_plotly, rotate_figure, format_2d_plotly

In [ ]:
# load embryo_df for our current best model
# root = "/media/nick/hdd02/Cole Trapnell's Lab Dropbox/Nick Lammers/Nick/morphseq/"

root = "/Users/nick/Cole Trapnell's Lab Dropbox/Nick Lammers/Nick/morphseq/"

# path to save data
read_path = os.path.join(root, "results", "20250312", "morph_latent_space", "")

# path to figures and data
fig_path = "/Users/nick/Cole Trapnell's Lab Dropbox/Nick Lammers/Nick/slides/morphseq/20250312/morph_metrics/"
fig_data_path = "/Users/nick/Cole Trapnell's Lab Dropbox/Nick Lammers/Nick/slides/morphseq/20250312/data/morph_metrics/"
os.makedirs(fig_path, exist_ok=True)
os.makedirs(fig_data_path, exist_ok=True)

In [ ]:
import joblib

# load datasets
hf_pca_df = pd.read_csv(os.path.join(read_path, "hf_pca_morph_df.csv"))
ref_pca_df = pd.read_csv(os.path.join(read_path, "ab_ref_pca_morph_df.csv"))
spline_df = pd.read_csv(os.path.join(read_path, "spline_morph_df_full.csv"))
spline_df["knot_index"] = spline_df.index
hf_pca_df = hf_pca_df.rename(columns={"mdl_stage_hpf":"stage"})
# Save the model to a file
# morph_stage_model = joblib.load(os.path.join(read_path, 'morph_stage_model.joblib'))
hf_pca_df.head()

In [ ]:
# hf_pca_df = hf_pca_df.rename(columns={"mdl_stage_hpf":"stage"})
stage_df = hf_pca_df.loc[:, ["temperature", "timepoint", "stage"]].groupby(["temperature", "timepoint"]).agg(
                            ["mean", "std"]).reset_index()

stage_df["temperature"] = np.round(stage_df["temperature"]).astype(int)
new_cols = ["_".join(col) if col[1]!='' else col[0] for col in stage_df.columns]

stage_df.columns= new_cols
stage_df = stage_df.pivot(columns="temperature", index="timepoint", 
                          values=["stage_mean", "stage_std"]).reset_index()
stage_df.columns = [
    f"{val}_{temp}" if val else str(temp)
    for val, temp in stage_df.columns.to_flat_index()
]


# interpolate
stage_vec = np.linspace(24, 36)
mean_cols = [col for col in stage_df.columns if "_mean" in col]

if "timepoint_" in stage_df.columns:
    stage_df = stage_df.set_index("timepoint_")
    
# Interpolate each "_mean" column onto stage_vec
interp_df = pd.DataFrame({
    col: np.interp(stage_vec, stage_df.index.values, stage_df[col].values)
    for col in mean_cols
})

# Optionally, add stage_vec as a column
interp_df["stage"] = stage_vec

interp_df.head()

In [ ]:
fig = go.Figure()

fig.add_traces(go.Scatter(x=interp_df["stage"], y=interp_df["stage_mean_28"], mode="lines", name="28C"))
fig.add_traces(go.Scatter(x=interp_df["stage"], y=interp_df["stage_mean_34"], mode="lines", name="34C"))
fig.add_traces(go.Scatter(x=interp_df["stage"], y=interp_df["stage_mean_35"], mode="lines", name="35C"))

fig.update_layout(
    width=800,   # width in pixels
    height=600   # height in pixels
)

xmin = 24
xmax=36
fig.update_layout(
    xaxis=dict(range=[24, 35]),
    yaxis=dict(range=[24, 38])
)

fig.add_shape(
    type="line",
    x0=xmin,  # or a fixed value like 0
    x1=xmax,  # or a fixed value like 100
    y0=26.9,
    y1=26.9,
    line=dict(color="black", width=2, dash="dash")
)

fig.show()

In [ ]:
stage_df.head()